[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/23.%20The%20AGV%20Dispatching%20%26%20Battery%20Management%20Problem/P23-Tier-7_executed.ipynb)

---


# Problem 23: The AGV Dispatching and Battery Management Problem

## Tier 7: Human–AI Symbiotic Partnership

### Purpose

This notebook simulates a control tower in which human operators set strategic goals and review selected recommendations, while an AI component analyzes operational indicators, ranks candidate actions, explains its recommendation, and detects anomalies. The example contains eight automated guided vehicles (AGVs), two human operators, and a 25-step simulation.

### Inputs and outputs

The simulation begins with a randomly initialized AGV fleet and operational metrics. At each five-minute step, it updates battery levels and system indicators, may generate transport tasks, and passes the resulting situation to the control tower. The notebook records decisions, operator interactions, alerts, and a performance timeline. It then displays a summary, visualizations, and descriptive collaboration metrics.

The main input is the simulated state assembled by `HumanAISimulation._create_situation_data`. The control tower returns an AI recommendation, any detected alerts, and the path taken by the recommendation: human review, autonomous execution, or queueing when no operator is available. The timeline and descriptive analysis summarize the resulting simulation; they are not measurements of a real terminal.

### Scope and modeling assumptions

This is an illustrative, stochastic simulation—not a validated AGV dispatch optimizer or a calibrated model of a real terminal. Candidate actions are evaluated with simulated random scores; no route network, travel-time model, charging capacity constraint, or task-to-vehicle assignment optimization is represented. Strategic goals are stored and supplied as context, but they do not mathematically constrain the action selection. Execution effects are simplified rules.

The feedback interface stores human feedback. The placeholder reinforcement and adjustment methods do not train or update a machine-learning model, and the simulation does not automatically generate feedback after each action. Likewise, the anomaly detection rate and pattern-recognition accuracy are configured illustrative indicators, not estimates computed from labeled observations. Interpret the reported values as properties of this demonstration, not evidence of operational effectiveness.

### Collaboration workflow

1. Update the simulated fleet and operational state.
2. Extract features and recognize threshold-based patterns.
3. Generate and score a small set of candidate actions.
4. Explain the selected action and decide whether human review is required.
5. Detect anomalies and route the decision to an available operator, or execute it autonomously.
6. Apply simplified effects, record the step, and summarize performance.

### Decision logic

For a situation with feature vector $x$, the AI constructs a candidate set $A(x)$. Each candidate $a \in A(x)$ receives a simulated score $s(a)$ and confidence value $c(a)$. The selected recommendation is

$$
a^* = \arg\max_{a \in A(x)} s(a).
$$

In the code, `_generate_alternatives` constructs $A(x)$, `_evaluate_alternatives` assigns the scores and confidence values, and `analyze_situation` selects the maximum-scoring candidate. Each score is sampled independently from a uniform distribution on $[0.6, 0.95]$. Confidence is the score plus a random offset from $[-0.1, 0.1]$, clipped to $[0,1]$. These values are demonstration inputs to the decision logic, not estimates learned from AGV operating data. Ties are resolved by Python's `max` operation, which selects the first candidate with the maximum score.

A human review is required when an intervention rule triggers or when the selected action's confidence is below the autonomous threshold of $0.8$. In this example, the rules request review for a safety risk above $0.9$, the presence of an anomaly, or an illustrative expected improvement above the configured high-value threshold. Expected improvement is represented on a 20-point scale as $20s(a)$, so the high-value comparison uses $20$ times its fractional threshold. The rules are explicit policy conditions; they do not form a learned classifier.

When review is required, the control tower selects an available operator by maximizing expertise multiplied by the fraction of workload capacity currently free. If $E_h$ is operator $h$'s expertise, $w_h$ is current workload, and $W_h$ is workload capacity, the selection score is

$$
q_h = E_h\left(1 - \frac{w_h}{W_h}\right), \qquad h \in H_{\mathrm{available}}.
$$

The operator's approval, modification, or override is simulated probabilistically. The operator is selected using `HumanControlTower._select_operator`; the response is generated by `_simulate_human_decision`. If no operator is available, the recommendation is added to `pending_decisions`. This is a transparent teaching model of a human-in-the-loop workflow, not a behavioral model of actual operators.

### Simulation state and reported indicators

A simulation step advances time by five minutes, updates AGV battery levels and system indicators, optionally adds tasks, and passes a snapshot to the control tower. For AGV $i$, the battery update is a bounded random change: a moving or servicing vehicle loses a random amount, a charging vehicle gains a random amount, and the result is kept within $[0,100]$. Fleet average battery is then recalculated. Efficiency, congestion, and time pressure are also updated with bounded random changes. These operations are implemented in `HumanAISimulation._update_system_state` and `_create_situation_data`.

The control tower's collaboration score is a constructed summary, not an independently measured outcome. After decision $t$, it is calculated as

$$
C_t = 0.7\frac{A_t}{t} + 0.3\left(1 - \frac{O_t}{t}\right), \qquad t > 0,
$$
where $A_t$ is the number of autonomous decisions and $O_t$ is the number of human overrides among the first $t$ decisions. The implementation updates these counts and computes the score in `HumanControlTower._update_metrics`. Queued decisions are included in the total decision count but are neither autonomous decisions nor overrides. Consequently, this score is best understood as a descriptive indicator defined by the example, not as a general measure of collaboration quality.

The simulation performs a bounded number of steps, and each step examines the fleet and the current task queue. If there are $T$ steps, $N$ AGVs, and at most $Q$ queued tasks per step, the state-update and task-inspection work grows on the order of $T(N+Q)$, apart from plotting and reporting. Candidate evaluation is linear in the number of candidates for each decision. This small simulation does not solve a combinatorial dispatch or routing optimization problem.

### What to examine

Follow how operational features lead to patterns, candidate actions, explanations, and intervention decisions. Then compare the decision distribution and time series. The charts describe this seeded simulation run; they do not establish that collaboration improves real-world decision quality. Consider which additional operational data, constraints, and outcome measurements would be necessary before making such a claim.

In [ ]:
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import random
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Fixed seeds make the random choices reproducible within the same software environment.
np.random.seed(42)
random.seed(42)

plt.style.use('default')
sns.set_palette('husl')

In [ ]:
'''Data records used by the control tower and its simulation.'''

@dataclass
class HumanOperator:
    '''Human operator with expertise and decision-making capacity.'''
    id: str
    name: str
    expertise_level: float
    workload_capacity: int
    current_workload: int = 0
    decisions_made: List[Dict[str, Any]] = field(default_factory=list)
    override_history: List[Dict[str, Any]] = field(default_factory=list)
    satisfaction_score: float = 0.8


@dataclass
class AIDecision:
    '''AI recommendation together with its explanation and evidence.'''
    decision_id: str
    decision_type: str
    recommended_action: str
    confidence: float
    reasoning: List[str]
    supporting_data: Dict[str, Any]
    alternatives: List[Dict[str, Any]] = field(default_factory=list)
    requires_human_approval: bool = False
    priority: str = 'medium'


@dataclass
class HumanAICollaboration:
    '''Record of a human decision about an AI recommendation.'''
    timestamp: datetime
    human_id: str
    ai_decision_id: str
    human_action: str
    final_decision: str
    reasoning: str
    outcome_score: float


@dataclass
class AnomalyAlert:
    '''An anomaly report requiring attention or follow-up.'''
    alert_id: str
    anomaly_type: str
    severity: str
    description: str
    affected_assets: List[str]
    recommended_actions: List[str]
    confidence: float
    timestamp: datetime = field(default_factory=datetime.now)
    status: str = 'pending'


@dataclass
class StrategicGoal:
    '''High-level objective supplied by a human operator.'''
    goal_id: str
    description: str
    priority: float
    target_metrics: Dict[str, float]
    time_horizon: timedelta
    set_by: str
    progress: float = 0.0
    status: str = 'active'


@dataclass
class LearningFeedback:
    '''Outcome feedback stored for later review by the AI component.'''
    feedback_id: str
    human_id: str
    original_ai_recommendation: str
    human_decision: str
    decision_outcome: float
    lesson_learned: str
    timestamp: datetime = field(default_factory=datetime.now)


@dataclass
class ControlTowerMetrics:
    '''Snapshot of collaboration and system-level indicators.'''
    timestamp: datetime
    total_decisions: int
    human_overrides: int
    ai_autonomous_decisions: int
    collaboration_score: float
    anomaly_detection_accuracy: float
    human_workload_balance: float
    system_learning_rate: float
    overall_performance: float

In [ ]:
class ExplainableAI:
    '''Threshold-based decision demonstration with human-readable explanations.'''

    def __init__(self, ai_id: str = 'XAI-Controller'):
        self.ai_id = ai_id
        self.decision_history: List[AIDecision] = []
        self.learning_database: List[LearningFeedback] = []
        self.anomaly_detector = AnomalyDetector()
        self.pattern_recognition_accuracy = 0.85
        self.prediction_confidence_threshold = 0.7
        self.explainability_level = 0.9
        self.autonomous_threshold = 0.8
        self.human_intervention_rules = self._load_intervention_rules()

    def _load_intervention_rules(self) -> List[Dict[str, Any]]:
        '''Define the conditions that request human review.'''
        return [
            {'condition': 'safety_critical', 'threshold': 0.9,
             'action': 'require_human_approval'},
            {'condition': 'novel_situation', 'threshold': 0.6,
             'action': 'recommend_human_review'},
            {'condition': 'high_value_decision', 'threshold': 0.7,
             'action': 'suggest_human_override'}
        ]

    def analyze_situation(self, situation_data: Dict[str, Any]) -> AIDecision:
        '''Create, explain, and classify one recommendation.'''
        features = self._extract_features(situation_data)
        patterns = self._recognize_patterns(features)
        alternatives = self._generate_alternatives(features, patterns)
        evaluated = self._evaluate_alternatives(alternatives)
        best_action = max(evaluated, key=lambda item: item['score'])
        explanation = self._generate_explanation(features, patterns, best_action)
        requires_human = self._requires_human_approval(features, best_action)

        decision = AIDecision(
            decision_id=f'AI-{len(self.decision_history) + 1}',
            decision_type=best_action['type'],
            recommended_action=best_action['action'],
            confidence=best_action['confidence'],
            reasoning=explanation,
            supporting_data=best_action['data'],
            alternatives=evaluated,
            requires_human_approval=requires_human,
            priority=self._determine_priority(best_action)
        )
        self.decision_history.append(decision)
        return decision

    def _extract_features(self, situation_data: Dict[str, Any]) -> Dict[str, Any]:
        '''Select the operational indicators used by the decision rules.'''
        return {
            'num_agvs': situation_data.get('num_agvs', 0),
            'num_tasks': situation_data.get('num_tasks', 0),
            'avg_battery': situation_data.get('avg_battery', 100),
            'system_efficiency': situation_data.get('efficiency', 0.8),
            'time_pressure': situation_data.get('time_pressure', 0.5),
            'congestion_level': situation_data.get('congestion', 0.3),
            'anomaly_indicators': situation_data.get('anomalies', []),
            'strategic_goals': situation_data.get('goals', []),
            'safety_risk': situation_data.get('safety_risk', 0)
        }

    def _recognize_patterns(self, features: Dict[str, Any]) -> Dict[str, Any]:
        '''Apply explicit thresholds to identify operating conditions.'''
        return {
            'high_demand': features['num_tasks'] > 10,
            'low_battery': features['avg_battery'] < 30,
            'congestion_detected': features['congestion_level'] > 0.7,
            'time_critical': features['time_pressure'] > 0.8,
            'anomaly_present': len(features['anomaly_indicators']) > 0,
            'efficiency_drop': features['system_efficiency'] < 0.6
        }

    def _generate_alternatives(
        self, features: Dict[str, Any], patterns: Dict[str, Any]
    ) -> List[Dict[str, Any]]:
        '''Build the candidate actions considered for the current situation.'''
        alternatives = [
            {'type': 'resource_allocation',
             'action': 'optimize_agv_assignment',
             'description': 'Rebalance AGV assignments for better efficiency'},
            {'type': 'battery_management',
             'action': 'prioritize_charging',
             'description': 'Send low-battery AGVs to charging stations'},
            {'type': 'congestion_control',
             'action': 'reroute_agvs',
             'description': 'Reroute AGVs to avoid congested areas'}
        ]
        if patterns['time_critical']:
            alternatives.append({
                'type': 'urgent_response',
                'action': 'expedite_critical_tasks',
                'description': 'Prioritize time-critical tasks'
            })
        if patterns['anomaly_present']:
            alternatives.append({
                'type': 'anomaly_response',
                'action': 'investigate_anomaly',
                'description': 'Investigate and resolve detected anomalies'
            })
        return alternatives

    def _evaluate_alternatives(
        self, alternatives: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        '''Attach illustrative random scores and supporting details.'''
        evaluated = []
        for alternative in alternatives:
            score = random.uniform(0.6, 0.95)
            confidence = min(1.0, max(0.0, score + random.uniform(-0.1, 0.1)))
            evaluated.append({
                **alternative,
                'score': score,
                'confidence': confidence,
                'data': {
                    'expected_improvement': score * 20,
                    'risk_level': 'low' if score > 0.8 else 'medium',
                    'implementation_time': random.randint(5, 30)
                }
            })
        return evaluated

    def _generate_explanation(
        self, features: Dict[str, Any], patterns: Dict[str, Any], action: Dict[str, Any]
    ) -> List[str]:
        '''Translate selected features and simulated estimates into readable reasons.'''
        explanation = [
            f'Recommended {action["description"]} with '
            f'{action["confidence"]:.1%} confidence.'
        ]
        active_patterns = [name for name, active in patterns.items() if active]
        if active_patterns:
            readable = ', '.join(active_patterns).replace('_', ' ').title()
            explanation.append(f'Detected patterns: {readable}.')
        if features['avg_battery'] < 50:
            explanation.append(
                f'Average battery is {features["avg_battery"]:.1f}%, '
                'so energy management warrants attention.'
            )
        if features['system_efficiency'] < 0.7:
            explanation.append(
                f'System efficiency is {features["system_efficiency"]:.1%}, '
                'below the illustrative 70% attention level.'
            )
        improvement = action['data']['expected_improvement']
        explanation.append(
            f'The simulated evaluation assigns an expected improvement of '
            f'approximately {improvement:.1f}%; this is not a measured forecast.'
        )
        explanation.append(
            f'Illustrative risk category: {action["data"]["risk_level"]}.'
        )
        return explanation

    def _requires_human_approval(
        self, features: Dict[str, Any], action: Dict[str, Any]
    ) -> bool:
        '''Require review when a rule triggers or confidence is insufficient.'''
        if any(self._evaluate_rule(rule, features, action)
               for rule in self.human_intervention_rules):
            return True
        if action['type'] in ['safety_critical', 'major_system_change']:
            return True
        return action['confidence'] < self.autonomous_threshold

    def _evaluate_rule(
        self, rule: Dict[str, Any], features: Dict[str, Any], action: Dict[str, Any]
    ) -> bool:
        '''Evaluate one intervention rule against the current features.'''
        condition = rule['condition']
        threshold = rule['threshold']
        if condition == 'safety_critical':
            return features.get('safety_risk', 0) > threshold
        if condition == 'novel_situation':
            return len(features.get('anomaly_indicators', [])) > 0
        if condition == 'high_value_decision':
            improvement_threshold = threshold * 20
            return action['data']['expected_improvement'] > improvement_threshold
        return False

    def _determine_priority(self, action: Dict[str, Any]) -> str:
        '''Assign a descriptive priority from action type and confidence.'''
        if action['type'] == 'anomaly_response':
            return 'critical'
        if action['type'] == 'urgent_response':
            return 'high'
        if action['confidence'] > 0.8:
            return 'medium'
        return 'low'

    def learn_from_feedback(self, feedback: LearningFeedback) -> None:
        '''Store feedback; model training is outside this demonstration's scope.'''
        self.learning_database.append(feedback)
        if feedback.decision_outcome > 0.8:
            self._reinforce_pattern(
                feedback.original_ai_recommendation, feedback.human_decision
            )
        elif feedback.decision_outcome < 0.5:
            self._adjust_pattern(
                feedback.original_ai_recommendation, feedback.human_decision
            )

    def _reinforce_pattern(self, ai_rec: str, human_dec: str) -> None:
        '''Placeholder for a model-update procedure in a deployed learning system.'''
        return None

    def _adjust_pattern(self, ai_rec: str, human_dec: str) -> None:
        '''Placeholder for a model-update procedure in a deployed learning system.'''
        return None


class AnomalyDetector:
    '''Detect a small set of threshold-defined operational anomalies.'''

    def __init__(self):
        self.baseline_metrics = {
            'avg_efficiency': 0.8,
            'avg_battery_level': 70,
            'avg_task_completion_time': 15,
            'communication_frequency': 10
        }
        self.anomaly_thresholds = {
            'efficiency_drop': 0.2,
            'battery_critical': 20,
            'task_delay': 30,
            'communication_silence': 5
        }

    def detect_anomalies(
        self, current_metrics: Dict[str, Any]
    ) -> List[AnomalyAlert]:
        '''Return alerts when supplied metrics cross the configured rules.'''
        anomalies = []
        efficiency = current_metrics.get('efficiency', 0)
        if efficiency < (
            self.baseline_metrics['avg_efficiency']
            - self.anomaly_thresholds['efficiency_drop']
        ):
            anomalies.append(AnomalyAlert(
                alert_id=f'ANOM-EFF-{len(anomalies) + 1}',
                anomaly_type='efficiency_drop',
                severity='high',
                description=f'System efficiency dropped to {efficiency:.1%}.',
                affected_assets=['all_agvs'],
                recommended_actions=[
                    'investigate_causes', 'optimize_routing', 'check_maintenance'
                ],
                confidence=0.9
            ))

        low_battery_agvs = current_metrics.get('low_battery_agvs', [])
        if len(low_battery_agvs) > 2:
            anomalies.append(AnomalyAlert(
                alert_id=f'ANOM-BAT-{len(anomalies) + 1}',
                anomaly_type='battery_critical',
                severity='critical',
                description=(
                    f'{len(low_battery_agvs)} AGVs have critically low batteries.'
                ),
                affected_assets=low_battery_agvs,
                recommended_actions=[
                    'prioritize_charging', 'reroute_tasks', 'activate_backup_agvs'
                ],
                confidence=0.95
            ))

        delayed_tasks = current_metrics.get('delayed_tasks', [])
        if len(delayed_tasks) > 3:
            anomalies.append(AnomalyAlert(
                alert_id=f'ANOM-DELAY-{len(anomalies) + 1}',
                anomaly_type='task_delays',
                severity='medium',
                description=(
                    f'{len(delayed_tasks)} tasks are experiencing significant delays.'
                ),
                affected_assets=delayed_tasks,
                recommended_actions=[
                    'reallocate_resources', 'extend_deadlines',
                    'prioritize_critical_tasks'
                ],
                confidence=0.8
            ))
        return anomalies

In [ ]:
class HumanControlTower:
    '''Coordinate recommendations, operator review, alerts, and reporting.'''

    def __init__(self):
        self.human_operators: Dict[str, HumanOperator] = {}
        self.ai_system = ExplainableAI()
        self.pending_decisions = deque()
        self.collaboration_history: List[HumanAICollaboration] = []
        self.strategic_goals: List[StrategicGoal] = []
        self.active_alerts: List[AnomalyAlert] = []
        self.metrics_history: List[ControlTowerMetrics] = []
        self._initialize_operators()
        self.performance_metrics = {
            'total_decisions': 0,
            'human_overrides': 0,
            'ai_autonomous': 0,
            'collaboration_score': 0.8,
            'anomaly_detection_rate': 0.9,
            'learning_effectiveness': 0.7
        }

    def _initialize_operators(self) -> None:
        '''Create the two operators used in the example.'''
        operators = [
            HumanOperator('HOP-1', 'Sarah Chen', 0.95, 5),
            HumanOperator('HOP-2', 'Marcus Johnson', 0.85, 4)
        ]
        self.human_operators = {operator.id: operator for operator in operators}
        print(f'Initialized {len(self.human_operators)} human operators.')

    def set_strategic_goal(
        self, description: str, priority: float,
        target_metrics: Dict[str, float], time_horizon: timedelta,
        operator_id: str
    ) -> StrategicGoal:
        '''Record a high-level objective supplied by an operator.'''
        if operator_id not in self.human_operators:
            raise ValueError(f'Unknown operator ID: {operator_id}')
        if not 0 <= priority <= 1:
            raise ValueError('Goal priority must be between 0 and 1.')
        goal = StrategicGoal(
            goal_id=f'GOAL-{len(self.strategic_goals) + 1}',
            description=description,
            priority=priority,
            target_metrics=target_metrics,
            time_horizon=time_horizon,
            set_by=operator_id
        )
        self.strategic_goals.append(goal)
        print(f'Strategic goal recorded: {description} (set by {operator_id}).')
        return goal

    def process_situation(self, situation_data: Dict[str, Any]) -> Dict[str, Any]:
        '''Analyze a situation, detect anomalies, then route its recommendation.'''
        anomalies = self.ai_system.anomaly_detector.detect_anomalies(situation_data)
        self.active_alerts.extend(anomalies)
        decision_data = dict(situation_data)
        decision_data['anomalies'] = list(situation_data.get('anomalies', []))
        decision_data['anomalies'].extend(
            alert.anomaly_type for alert in anomalies
        )
        decision_data['goals'] = [
            goal.description for goal in self.strategic_goals
            if goal.status == 'active'
        ]
        ai_decision = self.ai_system.analyze_situation(decision_data)

        if ai_decision.requires_human_approval:
            result = self._route_to_human(ai_decision, decision_data)
            collaboration_type = (
                'human_processed'
                if result['status'] == 'human_processed'
                else 'queued'
            )
        else:
            result = self._execute_autonomous(ai_decision, decision_data)
            collaboration_type = 'autonomous'

        self._update_metrics(ai_decision, result)
        return {
            'ai_decision': ai_decision,
            'anomalies': anomalies,
            'final_decision': result,
            'collaboration_type': collaboration_type
        }

    def _route_to_human(
        self, ai_decision: AIDecision, situation_data: Dict[str, Any]
    ) -> Dict[str, Any]:
        '''Send a review request to the best available operator.'''
        selected_operator = self._select_operator(ai_decision)
        if selected_operator is None:
            self.pending_decisions.append(ai_decision)
            return {
                'status': 'queued',
                'reason': 'no_available_operator',
                'decision': ai_decision
            }

        human_decision = self._simulate_human_decision(
            selected_operator, ai_decision, situation_data
        )
        collaboration = HumanAICollaboration(
            timestamp=datetime.now(),
            human_id=selected_operator.id,
            ai_decision_id=ai_decision.decision_id,
            human_action=human_decision['action'],
            final_decision=human_decision['final_action'],
            reasoning=human_decision['reasoning'],
            outcome_score=0.0
        )
        self.collaboration_history.append(collaboration)
        return {
            'status': 'human_processed',
            'operator': selected_operator.id,
            'human_decision': human_decision,
            'collaboration': collaboration
        }

    def _select_operator(
        self, ai_decision: AIDecision
    ) -> Optional[HumanOperator]:
        '''Choose an available operator using expertise and free capacity.'''
        available = [
            operator for operator in self.human_operators.values()
            if operator.current_workload < operator.workload_capacity
        ]
        if not available:
            return None
        return max(
            available,
            key=lambda operator: operator.expertise_level * (
                1 - operator.current_workload / operator.workload_capacity
            )
        )

    def _simulate_human_decision(
        self, operator: HumanOperator, ai_decision: AIDecision,
        situation_data: Dict[str, Any]
    ) -> Dict[str, Any]:
        '''Simulate an approve, modify, or override response.'''
        operator.current_workload += 1
        try:
            acceptance_probability = operator.expertise_level * ai_decision.confidence
            if random.random() < acceptance_probability:
                action = 'approve'
                final_action = ai_decision.recommended_action
                reasoning = (
                    'Accepted the recommendation based on its confidence '
                    'and stated reasoning.'
                )
            elif random.random() < 0.7:
                action = 'modify'
                if ai_decision.alternatives:
                    alternative = random.choice(ai_decision.alternatives)
                    final_action = alternative['action']
                    reasoning = (
                        'Selected a different candidate action after human review.'
                    )
                else:
                    final_action = ai_decision.recommended_action
                    reasoning = 'Retained the recommendation after human review.'
            else:
                action = 'override'
                final_action = 'manual_intervention_required'
                reasoning = (
                    'Overrode the recommendation because contextual factors '
                    'may not be represented in the analysis.'
                )
                operator.override_history.append({
                    'timestamp': datetime.now(),
                    'ai_decision': ai_decision.decision_id,
                    'ai_recommendation': ai_decision.recommended_action,
                    'human_decision': final_action,
                    'reason': reasoning
                })

            operator.decisions_made.append({
                'timestamp': datetime.now(),
                'ai_decision_id': ai_decision.decision_id,
                'action': action,
                'final_action': final_action,
                'reasoning': reasoning
            })
            return {
                'action': action,
                'final_action': final_action,
                'reasoning': reasoning,
                'operator_expertise': operator.expertise_level
            }
        finally:
            operator.current_workload = max(0, operator.current_workload - 1)

    def _execute_autonomous(
        self, ai_decision: AIDecision, situation_data: Dict[str, Any]
    ) -> Dict[str, Any]:
        '''Represent autonomous execution with a confidence-based outcome flag.'''
        success = ai_decision.confidence > 0.7
        return {
            'status': 'executed_autonomously',
            'action': ai_decision.recommended_action,
            'confidence': ai_decision.confidence,
            'success': success,
            'execution_time': random.randint(1, 10)
        }

    def _update_metrics(
        self, ai_decision: AIDecision, result: Dict[str, Any]
    ) -> None:
        '''Update counters and a descriptive collaboration score.'''
        self.performance_metrics['total_decisions'] += 1
        if result.get('status') == 'human_processed':
            human_action = result.get('human_decision', {}).get('action', '')
            if human_action == 'override':
                self.performance_metrics['human_overrides'] += 1
        elif result.get('status') == 'executed_autonomously':
            self.performance_metrics['ai_autonomous'] += 1

        total = self.performance_metrics['total_decisions']
        autonomous_ratio = self.performance_metrics['ai_autonomous'] / total
        override_ratio = self.performance_metrics['human_overrides'] / total
        self.performance_metrics['collaboration_score'] = (
            autonomous_ratio * 0.7 + (1 - override_ratio) * 0.3
        )

    def collect_feedback(
        self, collaboration_id: str, outcome_score: float, lesson: str
    ) -> bool:
        '''Attach outcome feedback to a collaboration identified by its timestamp.'''
        if not 0 <= outcome_score <= 1:
            raise ValueError('Outcome score must be between 0 and 1.')
        collaboration = next(
            (item for item in self.collaboration_history
             if str(item.timestamp) == collaboration_id),
            None
        )
        if collaboration is None:
            return False

        collaboration.outcome_score = outcome_score
        decision = next(
            (item for item in self.ai_system.decision_history
             if item.decision_id == collaboration.ai_decision_id),
            None
        )
        original_recommendation = (
            decision.recommended_action if decision is not None else ''
        )
        feedback = LearningFeedback(
            feedback_id=f'FEEDBACK-{len(self.ai_system.learning_database) + 1}',
            human_id=collaboration.human_id,
            original_ai_recommendation=original_recommendation,
            human_decision=collaboration.final_decision,
            decision_outcome=outcome_score,
            lesson_learned=lesson
        )
        self.ai_system.learn_from_feedback(feedback)
        self.performance_metrics['learning_effectiveness'] = min(
            1.0, self.performance_metrics['learning_effectiveness'] + 0.01
        )
        return True

    def get_status_report(self) -> Dict[str, Any]:
        '''Return operator, AI, and performance summaries.'''
        return {
            'operators': {
                operator.id: {
                    'name': operator.name,
                    'expertise': operator.expertise_level,
                    'workload': (
                        f'{operator.current_workload}/'
                        f'{operator.workload_capacity}'
                    ),
                    'decisions_made': len(operator.decisions_made),
                    'overrides': len(operator.override_history)
                }
                for operator in self.human_operators.values()
            },
            'ai_system': {
                'decisions_made': len(self.ai_system.decision_history),
                'learning_entries': len(self.ai_system.learning_database),
                'pattern_recognition': self.ai_system.pattern_recognition_accuracy
            },
            'performance': dict(self.performance_metrics),
            'active_alerts': len(self.active_alerts),
            'strategic_goals': len(self.strategic_goals),
            'pending_decisions': len(self.pending_decisions)
        }

In [ ]:
class HumanAISimulation:
    '''Simulate AGV operations and record human-AI interactions.'''

    def __init__(self):
        self.control_tower = HumanControlTower()
        self.current_time = datetime.now()
        self.simulation_step = 0
        self.max_steps = 50
        self.agv_fleet = self._initialize_agv_fleet()
        self.task_queue: List[Dict[str, Any]] = []
        self.system_metrics = {
            'efficiency': 0.8,
            'avg_battery': 75,
            'congestion': 0.3,
            'time_pressure': 0.5
        }
        self.decision_history: List[Dict[str, Any]] = []
        self.performance_timeline: List[Dict[str, Any]] = []

    def _initialize_agv_fleet(self) -> List[Dict[str, Any]]:
        '''Create eight AGVs with randomized initial operating states.'''
        fleet = []
        for index in range(8):
            fleet.append({
                'id': f'AGV-{index + 1}',
                'battery': random.uniform(20, 100),
                'status': random.choice(
                    ['idle', 'moving', 'charging', 'servicing']
                ),
                'location': (random.uniform(0, 10), random.uniform(0, 8)),
                'current_task': None
            })
        return fleet

    def simulate_step(self) -> None:
        '''Advance the simulation by one five-minute interval.'''
        self.simulation_step += 1
        self.current_time += timedelta(minutes=5)
        print(
            f'\n=== Step {self.simulation_step} - '
            f'{self.current_time.strftime("%H:%M")} ==='
        )
        self._update_system_state()
        if random.random() < 0.3:
            self._generate_tasks()

        situation_data = self._create_situation_data()
        result = self.control_tower.process_situation(situation_data)
        self._execute_decision(result)
        self.decision_history.append({
            'step': self.simulation_step,
            'timestamp': self.current_time,
            'result': result
        })
        self._update_performance_timeline()
        self._print_step_summary(result)

    def _update_system_state(self) -> None:
        '''Update batteries, status changes, and bounded system indicators.'''
        for agv in self.agv_fleet:
            if agv['status'] in ['moving', 'servicing']:
                agv['battery'] = max(
                    0, agv['battery'] - random.uniform(1, 3)
                )
            elif agv['status'] == 'charging':
                agv['battery'] = min(
                    100, agv['battery'] + random.uniform(2, 5)
                )
            if random.random() < 0.1:
                agv['status'] = random.choice(['idle', 'moving', 'servicing'])

        self.system_metrics['avg_battery'] = float(
            np.mean([agv['battery'] for agv in self.agv_fleet])
        )
        self.system_metrics['efficiency'] = max(
            0.4, min(0.95, self.system_metrics['efficiency']
                     + random.uniform(-0.05, 0.05))
        )
        self.system_metrics['congestion'] = max(
            0.1, min(0.9, self.system_metrics['congestion']
                     + random.uniform(-0.1, 0.1))
        )
        self.system_metrics['time_pressure'] = max(
            0.2, min(0.9, self.system_metrics['time_pressure']
                     + random.uniform(-0.1, 0.1))
        )

    def _generate_tasks(self) -> None:
        '''Add one to three transport tasks to the pending queue.'''
        num_tasks = random.randint(1, 3)
        for _ in range(num_tasks):
            task_number = len(self.task_queue) + 1
            self.task_queue.append({
                'id': f'TASK-{task_number}',
                'priority': random.uniform(0.5, 1.0),
                'pickup': f'P{random.randint(1, 4)}',
                'dropoff': f'D{random.randint(1, 4)}',
                'deadline': self.current_time + timedelta(
                    minutes=random.randint(20, 60)
                )
            })
        print(
            f'Generated {num_tasks} new tasks. '
            f'Total pending: {len(self.task_queue)}.'
        )

    def _create_situation_data(self) -> Dict[str, Any]:
        '''Assemble the operating state passed to the control tower.'''
        low_battery_agvs = [
            agv['id'] for agv in self.agv_fleet if agv['battery'] < 25
        ]
        delayed_tasks = [
            task['id'] for task in self.task_queue
            if task['deadline'] < self.current_time + timedelta(minutes=15)
        ]
        return {
            'num_agvs': len(self.agv_fleet),
            'num_tasks': len(self.task_queue),
            'avg_battery': self.system_metrics['avg_battery'],
            'efficiency': self.system_metrics['efficiency'],
            'congestion': self.system_metrics['congestion'],
            'time_pressure': self.system_metrics['time_pressure'],
            'low_battery_agvs': low_battery_agvs,
            'delayed_tasks': delayed_tasks,
            'anomalies': [],
            'goals': [
                goal.description for goal in self.control_tower.strategic_goals
            ]
        }

    def _execute_decision(self, result: Dict[str, Any]) -> None:
        '''Apply simplified effects for the selected human or AI action.'''
        collaboration_type = result.get('collaboration_type', 'unknown')
        if collaboration_type == 'human_processed':
            human_decision = result['final_decision']['human_decision']
            action = human_decision.get('final_action', 'no_action')
            if 'optimize' in action:
                self.system_metrics['efficiency'] = min(
                    0.95, self.system_metrics['efficiency'] + 0.05
                )
            elif 'charging' in action:
                for agv in self.agv_fleet:
                    if agv['battery'] < 30:
                        agv['status'] = 'charging'
            elif 'reroute' in action:
                self.system_metrics['congestion'] = max(
                    0.1, self.system_metrics['congestion'] - 0.1
                )
        elif collaboration_type == 'autonomous':
            autonomous_result = result['final_decision']
            action = autonomous_result['action']
            if autonomous_result['success']:
                if 'optimize' in action:
                    self.system_metrics['efficiency'] = min(
                        0.95, self.system_metrics['efficiency'] + 0.03
                    )
                elif 'charging' in action:
                    low_battery_agvs = [
                        agv for agv in self.agv_fleet if agv['battery'] < 30
                    ]
                    if low_battery_agvs:
                        low_battery_agvs[0]['status'] = 'charging'
                elif 'reroute' in action:
                    self.system_metrics['congestion'] = max(
                        0.1, self.system_metrics['congestion'] - 0.05
                    )

        if self.task_queue and random.random() < 0.2:
            self.task_queue.pop(0)

    def _update_performance_timeline(self) -> None:
        '''Save the indicators needed for the later plots and trend analysis.'''
        self.performance_timeline.append({
            'step': self.simulation_step,
            'timestamp': self.current_time,
            'efficiency': self.system_metrics['efficiency'],
            'avg_battery': self.system_metrics['avg_battery'],
            'congestion': self.system_metrics['congestion'],
            'pending_tasks': len(self.task_queue),
            'collaboration_score': (
                self.control_tower.performance_metrics['collaboration_score']
            )
        })

    def _print_step_summary(self, result: Dict[str, Any]) -> None:
        '''Print the recommendation and resulting collaboration path.'''
        collaboration_type = result.get('collaboration_type', 'unknown')
        ai_decision = result.get('ai_decision')
        print(f'  Collaboration type: {collaboration_type}')
        if ai_decision is not None:
            print(f'  AI recommendation: {ai_decision.recommended_action}')
            print(f'  AI confidence: {ai_decision.confidence:.1%}')
            print(f'  Priority: {ai_decision.priority}')
        if collaboration_type == 'human_processed':
            human_decision = result['final_decision']['human_decision']
            print(f'  Human action: {human_decision["action"]}')
            print(f'  Final decision: {human_decision["final_action"]}')
        print(f'  System efficiency: {self.system_metrics["efficiency"]:.1%}')
        print(f'  Average battery: {self.system_metrics["avg_battery"]:.1f}%')
        print(f'  Pending tasks: {len(self.task_queue)}')

    def run_simulation(self, steps: int = 30) -> List[Dict[str, Any]]:
        '''Set the example's goals, run steps, and print a final report.'''
        if steps < 0:
            raise ValueError('The number of steps cannot be negative.')
        print('=' * 60)
        print('HUMAN-AI SYMBIOTIC SYSTEM SIMULATION')
        print('=' * 60)
        if not self.control_tower.strategic_goals:
            self.control_tower.set_strategic_goal(
                'Maintain system efficiency above 85%', 0.9,
                {'efficiency': 0.85}, timedelta(hours=8), 'HOP-1'
            )
            self.control_tower.set_strategic_goal(
                'Minimize battery-related disruptions', 0.8,
                {'low_battery_incidents': 2}, timedelta(hours=4), 'HOP-2'
            )
        remaining_steps = min(steps, self.max_steps - self.simulation_step)
        for _ in range(max(0, remaining_steps)):
            self.simulate_step()
        self._print_final_summary()
        return self.performance_timeline

    def _print_final_summary(self) -> None:
        '''Print a descriptive summary of the current simulation state.'''
        print('\n' + '=' * 60)
        print('SIMULATION COMPLETE - SUMMARY')
        print('=' * 60)
        status = self.control_tower.get_status_report()
        print('\nHuman operator activity:')
        for operator_id, data in status['operators'].items():
            print(f'  {data["name"]} ({operator_id}):')
            print(f'    Expertise: {data["expertise"]:.1%}')
            print(f'    Decisions made: {data["decisions_made"]}')
            print(f'    Overrides: {data["overrides"]}')
            print(f'    Current workload: {data["workload"]}')

        print('\nAI system activity:')
        ai_data = status['ai_system']
        print(f'  Decisions made: {ai_data["decisions_made"]}')
        print(f'  Stored feedback entries: {ai_data["learning_entries"]}')
        print(
            '  Configured pattern-recognition indicator: '
            f'{ai_data["pattern_recognition"]:.1%}'
        )

        print('\nCollaboration counters:')
        performance = status['performance']
        total = performance['total_decisions']
        autonomous = performance['ai_autonomous']
        overrides = performance['human_overrides']
        print(f'  Total decisions: {total}')
        if total:
            print(f'  Autonomous decisions: {autonomous} ({autonomous / total:.1%})')
            print(f'  Human overrides: {overrides} ({overrides / total:.1%})')
        else:
            print('  Autonomous decisions: 0 (not applicable; no decisions recorded)')
            print('  Human overrides: 0 (not applicable; no decisions recorded)')
        print(f'  Illustrative collaboration score: '
              f'{performance["collaboration_score"]:.3f}')
        print(f'  Feedback indicator: '
              f'{performance["learning_effectiveness"]:.3f}')

        print('\nFinal simulated system state:')
        print(f'  System efficiency: {self.system_metrics["efficiency"]:.1%}')
        print(f'  Average battery: {self.system_metrics["avg_battery"]:.1f}%')
        print(f'  Congestion level: {self.system_metrics["congestion"]:.1%}')
        print(f'  Pending tasks: {len(self.task_queue)}')

    def visualize_results(self):
        '''Plot operational indicators and the distribution of decision paths.'''
        if not self.performance_timeline:
            print('No simulation data are available to plot.')
            return None

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(
            'Human-AI Symbiotic System: Simulation Performance',
            fontsize=14, fontweight='bold'
        )
        steps = [item['step'] for item in self.performance_timeline]
        efficiency = [item['efficiency'] for item in self.performance_timeline]
        battery = [item['avg_battery'] for item in self.performance_timeline]
        congestion = [item['congestion'] for item in self.performance_timeline]
        tasks = [item['pending_tasks'] for item in self.performance_timeline]
        collaboration = [
            item['collaboration_score'] for item in self.performance_timeline
        ]

        ax1 = axes[0, 0]
        ax1_twin = ax1.twinx()
        line1 = ax1.plot(
            steps, efficiency, color='blue', linewidth=2,
            label='System efficiency'
        )
        line2 = ax1_twin.plot(
            steps, collaboration, color='red', linewidth=2,
            label='Illustrative collaboration score'
        )
        ax1.set_xlabel('Simulation step')
        ax1.set_ylabel('System efficiency (fraction)')
        ax1_twin.set_ylabel('Collaboration score (fraction)')
        ax1.set_title('Efficiency and collaboration indicator')
        ax1.grid(True, alpha=0.3)
        lines = line1 + line2
        ax1.legend(lines, [line.get_label() for line in lines], loc='best')

        ax2 = axes[0, 1]
        ax2_twin = ax2.twinx()
        line1 = ax2.plot(
            steps, battery, color='green', linewidth=2,
            label='Average battery'
        )
        line2 = ax2_twin.plot(
            steps, congestion, color='orange', linewidth=2,
            label='Congestion level'
        )
        ax2.set_xlabel('Simulation step')
        ax2.set_ylabel('Average battery (%)')
        ax2_twin.set_ylabel('Congestion level (fraction)')
        ax2.set_title('Battery and congestion indicators')
        ax2.set_ylim(0, 100)
        ax2.grid(True, alpha=0.3)
        lines = line1 + line2
        ax2.legend(lines, [line.get_label() for line in lines], loc='best')

        ax3 = axes[1, 0]
        decision_types = {
            'Autonomous': 0,
            'Human approved': 0,
            'Human modified': 0,
            'Human override': 0,
            'Queued': 0
        }
        for item in self.decision_history:
            result = item['result']
            kind = result.get('collaboration_type', '')
            if kind == 'autonomous':
                decision_types['Autonomous'] += 1
            elif kind == 'queued':
                decision_types['Queued'] += 1
            elif kind == 'human_processed':
                action = result['final_decision']['human_decision']['action']
                if action == 'approve':
                    decision_types['Human approved'] += 1
                elif action == 'modify':
                    decision_types['Human modified'] += 1
                elif action == 'override':
                    decision_types['Human override'] += 1
        categories = list(decision_types)
        values = list(decision_types.values())
        bars = ax3.bar(
            categories, values,
            color=['lightblue', 'lightgreen', 'gold', 'salmon', 'gray'],
            alpha=0.8
        )
        ax3.set_xlabel('Decision path')
        ax3.set_ylabel('Number of decisions')
        ax3.set_title('Decision-path distribution')
        ax3.tick_params(axis='x', rotation=35)
        ax3.grid(True, axis='y', alpha=0.3)
        for bar, value in zip(bars, values):
            ax3.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(), str(value), ha='center', va='bottom'
            )

        ax4 = axes[1, 1]
        ax4.plot(
            steps, tasks, color='purple', linewidth=2, marker='o',
            markersize=4, label='Pending tasks'
        )
        ax4.set_xlabel('Simulation step')
        ax4.set_ylabel('Number of pending tasks')
        ax4.set_title('Task queue')
        ax4.grid(True, alpha=0.3)
        ax4.fill_between(steps, 0, tasks, alpha=0.2, color='purple')
        ax4.legend()

        fig.tight_layout()
        plt.show()
        return fig

In [ ]:
print('Initializing the human-AI symbiotic system...')
simulation = HumanAISimulation()
timeline = simulation.run_simulation(steps=25)
fig = simulation.visualize_results()

In [ ]:
def analyze_collaboration_effectiveness(simulation: HumanAISimulation) -> Dict[str, Any]:
    '''Summarize decision paths, operator activity, and simulated trends.'''
    print('\n' + '=' * 60)
    print('HUMAN-AI COLLABORATION: DESCRIPTIVE ANALYSIS')
    print('=' * 60)

    total_decisions = len(simulation.decision_history)
    counts = {
        'autonomous': 0,
        'human_approved': 0,
        'human_modified': 0,
        'human_override': 0,
        'queued': 0
    }
    for item in simulation.decision_history:
        result = item['result']
        kind = result.get('collaboration_type', '')
        if kind == 'autonomous':
            counts['autonomous'] += 1
        elif kind == 'queued':
            counts['queued'] += 1
        elif kind == 'human_processed':
            action = result['final_decision']['human_decision']['action']
            if action == 'approve':
                counts['human_approved'] += 1
            elif action == 'modify':
                counts['human_modified'] += 1
            elif action == 'override':
                counts['human_override'] += 1

    print('\n1. Decision-path distribution')
    print(f'  Total decisions: {total_decisions}')
    for name, count in counts.items():
        ratio = count / total_decisions if total_decisions else 0.0
        print(f'  {name.replace("_", " ").title()}: {count} ({ratio:.1%})')

    autonomous_ratio = (
        counts['autonomous'] / total_decisions if total_decisions else 0.0
    )
    target_autonomous_ratio = 0.7
    balance_score = 1.0 - abs(autonomous_ratio - target_autonomous_ratio)
    print(
        '  Illustrative balance score: '
        f'{balance_score:.3f} (distance from a chosen 70% autonomy reference)'
    )

    status = simulation.control_tower.get_status_report()
    print('\n2. Human operator activity')
    for operator_data in status['operators'].values():
        decisions = operator_data['decisions_made']
        overrides = operator_data['overrides']
        override_rate = overrides / decisions if decisions else 0.0
        print(f'  {operator_data["name"]}:')
        print(f'    Expertise setting: {operator_data["expertise"]:.1%}')
        print(f'    Decisions recorded: {decisions}')
        print(f'    Override rate: {override_rate:.1%}')

    print('\n3. Feedback and configured AI indicators')
    ai_data = status['ai_system']
    learning_entries = ai_data['learning_entries']
    learning_rate = (
        learning_entries / total_decisions if total_decisions else 0.0
    )
    print(f'  Stored feedback entries: {learning_entries}')
    print(
        '  Configured pattern-recognition indicator: '
        f'{ai_data["pattern_recognition"]:.1%}'
    )
    print(
        '  Feedback entries per decision: '
        f'{learning_rate:.3f} (feedback is not generated automatically)'
    )
    print(
        '  Feedback effectiveness setting: '
        f'{status["performance"]["learning_effectiveness"]:.3f}'
    )

    stability_score = 0.8
    efficiency_trend = 0.0
    collaboration_trend = 0.0
    if simulation.performance_timeline:
        first = simulation.performance_timeline[0]
        last = simulation.performance_timeline[-1]
        efficiency_trend = last['efficiency'] - first['efficiency']
        collaboration_trend = (
            last['collaboration_score'] - first['collaboration_score']
        )
        efficiencies = [
            point['efficiency'] for point in simulation.performance_timeline
        ]
        efficiency_variance = float(np.var(efficiencies))
        stability_score = 1.0 - min(efficiency_variance, 1.0)

    print('\n4. Simulated performance trends')
    print(f'  Efficiency change: {efficiency_trend:+.1%}')
    print(f'  Collaboration-score change: {collaboration_trend:+.3f}')
    print(
        f'  Descriptive efficiency stability indicator: {stability_score:.3f}'
    )

    print('\n5. Alerts and configured detection indicator')
    active_alerts = status['active_alerts']
    anomaly_detection_rate = status['performance']['anomaly_detection_rate']
    print(f'  Alerts recorded: {active_alerts}')
    print(
        '  Configured anomaly-detection indicator: '
        f'{anomaly_detection_rate:.1%} (not measured against labeled events)'
    )

    efficiency_score = (
        simulation.performance_timeline[-1]['efficiency']
        if simulation.performance_timeline else 0.8
    )
    weights = {
        'balance': 0.25,
        'performance': 0.20,
        'learning': 0.20,
        'stability': 0.15,
        'anomaly_detection': 0.10,
        'efficiency': 0.10
    }
    scores = {
        'balance': balance_score,
        'performance': status['performance']['collaboration_score'],
        'learning': status['performance']['learning_effectiveness'],
        'stability': stability_score,
        'anomaly_detection': anomaly_detection_rate,
        'efficiency': efficiency_score
    }
    overall_score = sum(weights[key] * scores[key] for key in weights)
    print('\n6. Illustrative composite score')
    print(
        f'  Weighted score: {overall_score:.3f}. '
        'This combines demonstration indicators and is not a validated '
        'measure of operational success.'
    )
    print('  Component contributions:')
    for key, score in scores.items():
        contribution = weights[key] * score
        print(
            f'    {key.replace("_", " ").title()}: {score:.3f} '
            f'(weight: {weights[key]:.2f}, contribution: {contribution:.3f})'
        )

    return {
        'overall_score': overall_score,
        'balance_score': balance_score,
        'learning_effectiveness': status['performance']['learning_effectiveness'],
        'collaboration_score': status['performance']['collaboration_score'],
        'decision_counts': counts,
        'efficiency_trend': efficiency_trend
    }


collaboration_analysis = analyze_collaboration_effectiveness(simulation)

## Summary and interpretation

The notebook demonstrates a human–AI control-tower workflow for an eight-AGV fleet and two operators. Its central ideas are explicit recommendation explanations, confidence-based routing to human review, simulated approval or override, threshold-based anomaly alerts, and the recording of strategic goals and feedback.

The decision-path chart distinguishes autonomous decisions, human approvals, modifications, overrides, and queued requests. The time-series plots show the simulated efficiency, battery, congestion, task queue, and a constructed collaboration indicator. These are outputs of the model's assumptions and random choices; they should be interpreted as a way to inspect the simulation logic, not as proof of improved decision quality or system performance.

The feedback records provide an interface for documenting outcomes and lessons. In this notebook, storing feedback does not modify the decision rules or train a predictive model. Similarly, the configured accuracy indicators are illustrative constants. A real evaluation would require operational observations, outcome labels, clearly defined metrics, and validation against those data.

### Modeling limitations

- AGV locations and tasks are generated, but no physical routing or task-assignment optimization is solved.
- Candidate action scores and operator responses are simulated rather than estimated from observed data.
- Strategic goals are recorded for context but are not enforced as mathematical constraints.
- The effects of actions are represented by simple rules, and the anomaly detector checks only the indicators supplied to it.
- The composite score uses chosen weights and demonstration indicators; it is not a universally meaningful measure of collaboration quality.

These boundaries make the example useful for studying the flow of human–AI interaction while keeping the distinction between a workflow simulation and an operational decision-support system explicit.